<a href="https://colab.research.google.com/github/nyee88/tvam_resin_mini/blob/main/notebooks/01_working_curve.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- Install minimal deps (fast) ---
!pip -q install pandas matplotlib

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# --- 1) Load your tiny Ec/Dp table ---
csv_path = "data/ec_dp_405nm.csv"  # when saved back to GitHub, this relative path will work
df = pd.read_csv(csv_path, comment='#')

# Basic sanity checks
required = ["resin_or_source", "Ec_mJ_cm2", "Dp_um", "wavelength_nm"]
missing = [c for c in required if c not in df.columns]
assert not missing, f"Missing columns: {missing}"
assert (df["wavelength_nm"] == 405).all(), "Keep wavelength at 405 nm for now."

display(df)

# --- 2) Quick plot: Dp vs Ec (both matter for exposure) ---
plt.figure()
plt.scatter(df["Ec_mJ_cm2"], df["Dp_um"])
plt.xlabel("Ec (mJ/cm²)")
plt.ylabel("Dp (µm)")
plt.title("Quick look: Dp vs Ec at 405 nm")
plt.show()

# --- 3) Tiny demo: dose needed for a target cured depth ---
# Jacobs working-curve: Cd = Dp * ln(E / Ec)  -> E = Ec * exp(Cd/Dp)
target_Cd_um = 100  # e.g., aim to fully cure ~100 µm
df["E_needed_mJcm2_for_100um"] = df.apply(
    lambda r: r["Ec_mJ_cm2"] * (2.718281828 ** (target_Cd_um / r["Dp_um"])),
    axis=1
)
display(df[["resin_or_source", "Ec_mJ_cm2", "Dp_um", "E_needed_mJcm2_for_100um"]])
